# **A Basic Agent that Answers a Task using an LLM**



**STEP 1:**
Installs CrewAI with proper Version

In [1]:
!pip uninstall crewai -y
!pip install crewai==0.64.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of embedchain to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of litellm to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of litellm to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━

**STEP 2:** Install CrewAI and LLM

In [1]:
!pip uninstall -y crewai litellm
!pip install crewai crewai-tools litellm

Found existing installation: crewai 0.64.0
Uninstalling crewai-0.64.0:
  Successfully uninstalled crewai-0.64.0
Found existing installation: litellm 1.80.0
Uninstalling litellm-1.80.0:
  Successfully uninstalled litellm-1.80.0
  Using cached litellm-1.82.0-py3-none-any.whl.metadata (30 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.3/89.3 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 839.1 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.4 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of instructor to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 4.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opentelemetry-exporter-otlp-proto-gr

**STEP 3:**
Importing Statements



In [2]:
import os
import io
import logging
import warnings
import contextlib
from crewai import Agent, Task, Crew
from crewai.llm import LLM
from crewai_tools import SerperDevTool

**EXPLANATION:**

* import io → Used for handling in-memory input and output operations.

*  import logging → Used to track logs and debug execution details.

*  import warnings → Used to control or suppress warning messages.

*  import contextlib → Used to manage temporary execution contexts safely.
*  from crewai import Agent, Task, Crew → Imports core CrewAI components to create agents, assign tasks, and manage workflows.


*  from crewai.llm import LLM → Used to configure and connect the language model (like GROQ).


*  from crewai_tools import SerperDevTool → Imports the Serper search tool for real-time web search integration.

**STEP 4:** Setting API Keys

In [3]:
#set API key
os.environ["GROQ_API_KEY"] = "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"
os.environ["SERPER_API_KEY"] = "d939068f8db1975321a0a7276eaa152371c4128d"
os.environ.pop("OPENAI_API_KEY", None)

**EXPLANATION:**

* Sets your Groq API key so the LLM can authenticate and work.
* Sets Serper API key to enable web search inside agents.

*  os.environ.pop("OPENAI_API_KEY", None) → Removes OpenAI key if it exists, so the system doesn’t accidentally use OpenAI instead of Groq.

**STEP 5:** Import & Tool Setup


In [4]:
# Add your keys here
os.environ["GROQ_API_KEY"] = "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"
os.environ["SERPER_API_KEY"] = "d939068f8db1975321a0a7276eaa152371c4128d"

# Disable OpenAI
os.environ.pop("OPENAI_API_KEY", None)

# Reduce logs
os.environ["LITELLM_LOG"] = "CRITICAL"
logging.getLogger("LiteLLM").setLevel(logging.CRITICAL)
os.environ["CREWAI_TRACING_ENABLED"] = "false"

# Suppress all warnings
warnings.filterwarnings("ignore")

# LLM (Groq)
llm = LLM(
    model="groq/llama-3.3-70b-versatile",
    temperature=0.5
)

# Serper Tool
search_tool = SerperDevTool()

**EXPLANATION:**

*  Creates a Groq LLM instance using Groq.
* Selects the LLaMA 70B versatile model for generating responses.

*  temperature=0.3 → Controls creativity (low value = more accurate & consistent output).
*  search_tool = SerperDevTool() → Initializes web search tool so agents can fetch real-time information.

**STEP 6:** Agent Creation

In [5]:
# Agent
research_agent = Agent(
    role="AI Research Assistant",
    goal="Search the internet and provide accurate answers.",
    backstory="You are an intelligent AI researcher who uses live internet data.",
    tools=[search_tool],
    llm=llm,
    verbose=True
)

**EXPLANATION:**


*  Agent(...) → Creates an AI agent.
* role → Defines the agent’s job (Web Research Assistant).

*  goal → Specifies what the agent must achieve (search & summarize).
*  goal → Specifies what the agent must achieve (search & summarize).

*  backstory → Guides agent behavior and response style.
*   tools → Enables web search capability
*  llm → Connects Groq LLM for generating responses.
* verbose=True → Displays detailed execution logs.

**STEP 7:** Task Creation

In [6]:
# Task (FIXED)
task = Task(
    description="Find the latest AI trends in 2026 and summarize them clearly.",
    expected_output="A clear structured summary of the latest AI trends in 2026.",
    agent=research_agent
)

**EXPLANATION:**
*   Task(...) → Creates a task for the agent.

*  description → Defines what needs to be done.

*   expected_output → Specifies the output format required.
*  agent=agent → Assigns the task to the created agent.


**STEP 8:** Crew Creation

In [7]:
# Crew
crew = Crew(
    agents=[research_agent],
    tasks=[task],
    verbose=False
)

**EXPLANATION:**

*  Crew(...) → Creates a team to manage workflow.
*  agents → Adds agent(s) to the crew.

*  tasks → Assigns task(s) to be executed.
*  verbose=True → Shows detailed execution process.

**STEP 9:** Execution

In [9]:
# Crew
crew = Crew(
    agents=[research_agent],
    tasks=[task],
    verbose=False
)

# Run
f = io.StringIO()

with contextlib.redirect_stdout(f):
    result = crew.kickoff()

print("========== FINAL OUTPUT ==========")
print(result)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Research Assistant                                                                                   │
│                                                                                                                 │
│  Task: Find the latest AI trends in 2026 and summarize them clearly.                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Research Assistant                                                                                   │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The latest AI trends in 2026 include several key developments that are expected to shape the industry and      │
│  have a significant impact on various aspects of business and society. Some of the top trends include:          │
│                                                                                                                 │
│  1. Agentic AI Orchestration: This trend refers to the use of AI to orchestrate complex, end-to-end workflows   │
│  semi-autonomously. This is expected to be a major opportunity for enterprises struggling with speed-to-value.  │
│                                                                                                                 │
│  2. AI Coding: AI coding is the use of artificial intelligence to generate code, which is expected to increase  │
│  productivity and efficiency in software development.                                                           │
│                                                                                                                 │
│  3. Scaling Valuable AI: This trend refers to the ability to scale AI solutions to meet the needs of large      │
│  organizations, which is critical for maximizing the value of AI investments.                                   │
│                                                                                                                 │
│  4. Security, Governance, and Controls: As AI becomes more pervasive, security, governance, and controls will   │
│  become increasingly important to ensure that AI systems are secure, transparent, and accountable.              │
│                                                                                                                 │
│  5. AI Agents: AI agents are expected to increase productivity for employees at all levels by automating        │
│  routine tasks and providing personalized support.                                                              │
│                                                                                                                 │
│  6. AI Infrastructure: AI infrastructure is becoming smarter, with the ability to learn and adapt to changing   │
│  conditions, which is critical for supporting the growing demand for AI services.                               │
│                                                                                                                 │
│  7. GenAI App Integration: More GenAI app integration is expected, which will enable organizations to leverage  │
│  the power of AI to drive business innovation and growth.                                                       │
│                                                                                                                 │
│  8. Advanced Multimodal AI: More advanced multimodal AI is expected, which will enable organizations to         │
│  interact with customers and employees in more natural and intuitive ways.                                      │
│                                                                                                                 │
│  9. AI-Driven Decision Automation: AI-driven decision automation is expected to become more prevalent, which    │
│  will enable organizations to make faster and more accurate decisions.                                          │
│                                                        

========== FINAL OUTPUT ==========
The latest AI trends in 2026 include several key developments that are expected to shape the industry and have a significant impact on various aspects of business and society. Some of the top trends include:

1. Agentic AI Orchestration: This trend refers to the use of AI to orchestrate complex, end-to-end workflows semi-autonomously. This is expected to be a major opportunity for enterprises struggling with speed-to-value.

2. AI Coding: AI coding is the use of artificial intelligence to generate code, which is expected to increase productivity and efficiency in software development.

3. Scaling Valuable AI: This trend refers to the ability to scale AI solutions to meet the needs of large organizations, which is critical for maximizing the value of AI investments.

4. Security, Governance, and Controls: As AI becomes more pervasive, security, governance, and controls will become increasingly important to ensure that AI systems are secure, transparent

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 

**EXPLANATION:**
*  agents=[research_agent] → Adds the research agent to the crew.
*  tasks=[task] → Assigns the task to be executed.
*  verbose=False → Hides detailed execution logs (clean output).
*  crew.kickoff() → Starts the workflow execution.
*  result → Stores the generated output.